In [0]:
dbutils.widgets.removeAll()

In [0]:
import os
import sys
import shutil
from pathlib import Path
import json

# Force local file system synchronization
os.sync()

# Absolute workspace configuration
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
%%capture
pip install pyedi

In [0]:
from Shared.EDIProcessing import EDIProcessor, CSVConverter
from dimProvider.EDIProcessing.mapper import Mapper

In [0]:
def move_file(src_path: Path, target_dir: Path) -> Path:
    """Moves a file to a target directory cleanly, ensuring the directory exists."""
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    shutil.move(str(src_path), str(target_path))
    return target_path

In [0]:
def process_single_file(incoming_file_path: Path, base_source_dir: Path) -> tuple:
    """Parses an EDI provider file, extracts metadata, maps records, and generates a targeted CSV."""
    active_file_path = incoming_file_path
    try:
        if not active_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {active_file_path}")
        
        active_file_path = move_file(active_file_path, base_source_dir / "inprogress")

        # Core Parsing & Domain Mapping
        structured_json = EDIProcessor().parse(str(active_file_path))
        
        # Metadata Extraction
        interchange = structured_json.get('interchange', {})
        client_id = interchange.get('sender_id', '').strip()
        file_id = interchange.get('control_number', '').strip()
        
        st_segment = structured_json.get('heading', {}).get('transaction_set_header_loop', {}).get('transaction_set_header_ST', {})
        layout_id = st_segment.get('transaction_set_identifier_code', 'PROVIDER').strip()
        
        # Isolated CSV Delivery Production Rules
        target_csv_name = f"{active_file_path.stem}.csv"
        target_csv_path = ROOT_DIR / "temp" / layout_id / target_csv_name
        target_csv_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Code Conversion Execution
        CSVConverter().converter(Mapper().map_provider(structured_json), str(target_csv_path))
        
        move_file(active_file_path, base_source_dir / "processed")
        
        print("client id: ", client_id, " file id: ", file_id, " layout id: ", layout_id, " csv name: ", target_csv_name)
        return client_id, file_id, layout_id, target_csv_name
        
    except Exception as e:
        print(f"Failed processing {active_file_path.name}: {e}")
        if active_file_path.exists():
            move_file(active_file_path, base_source_dir / "failed")
        raise

In [0]:
def build_payloads(processed_files: list) -> tuple:
    """Generates precise production payloads for downstream orchestration stages."""
    process_list = []
    consolidation_list = []
    
    for f in processed_files:
        specific_client_container = f"{ROOT_DIR}/temp/{f['layout_id']}"
        
        process_list.append({
            "ClientID": f['client_id'],
            "FileID": f['file_id'],
            "FileName": f['csv_filename'],
            "ClientContainer": specific_client_container,
            "CurrentFolderPath": "",
            "ProcessedFolderPath": "/Volumes/claimspan/bronze/provider",
            "ColumnDelimiter": ",",
            "HasHeader": "true",
            "IgnoreHeader": "False",
            "FileLayoutID": f['layout_id'],
            "FileLayoutDescription": f"Standard{f['layout_id']}",
            "SchemaFileName": "provider_hierarchy_schema.json",
            "SchemaFilePath": f"{ROOT_DIR}/dimProvider/Bronze/Schema",
            "TextQualifier": "\""
        })
        
        consolidation_list.append({
            "DataGroupTrackingID": f"TRACK_PROVIDER_{f['layout_id']}_{f['file_id']}",
            "DataGroupMappingId": f"{f['layout_id']}PROVIDER",
            "FileId": f['file_id'],
            "FileLayoutID": f['layout_id'],
            "FileLayoutDescription": f"Standard{f['layout_id']}",
            "CurrentContainer": "/Volumes/claimspan/bronze/provider",
            "CurrentFolderPath": "",
            "ConsolidatedMappingFilePath": f"{ROOT_DIR}/dimProvider/Bronze/Schema",
            "ConsolidatedMappingFileName": "provider_hierarchy_schema.json",
            "ConsolidatedLayerDataModelFilePath": f"{ROOT_DIR}/dimProvider/Bronze/Schema",
            "ConsolidatedLayerDataModel": "provider_hierarchy_schema.json",
            "ConsolidatedFolderPath": "/Volumes/claimspan/bronze/provider_consolidated"
        })
        
    return json.dumps({"FileIds": process_list}), json.dumps({"FileIds": consolidation_list})

In [0]:
def trigger_silver_notebooks():
    """Triggers Silver layer notebooks in the correct dependency order."""
    silver_notebooks_base = f"{ROOT_DIR}/dimProvider/Silver/Notebooks"
    
    try:
        # Step 1: Create Provider Person Bridge (must run first for linkage)
        print("\n=== Triggering ProviderPersonBridge (Silver Layer) ===")
        dbutils.notebook.run(f"{silver_notebooks_base}/ProviderPersonBridge", 600)
        print("ProviderPersonBridge completed successfully")
        
        # Step 2: Process Provider data (depends on bridge)
        print("\n=== Triggering Provider (Silver Layer) ===")
        dbutils.notebook.run(f"{silver_notebooks_base}/Provider", 600)
        print("Provider completed successfully")
        
        # Step 3: Process Provider Hierarchy
        print("\n=== Triggering ProviderHierarchy (Silver Layer) ===")
        dbutils.notebook.run(f"{silver_notebooks_base}/ProviderHierarchy", 600)
        print("ProviderHierarchy completed successfully")
        
        print("\nAll Silver layer notebooks completed successfully\n")
    except Exception as e:
        print(f"Silver layer processing failed: {e}")
        raise

In [0]:
def build_gold_payload(processed_files: list) -> str:
    """Generates Gold layer payload for GenericSubGroupProcessing."""
    if not processed_files:
        return json.dumps({"FileIds": []})
    
    # Use the first processed file for metadata (assuming single batch)
    first_file = processed_files[0]
    
    gold_entry = {
        "FileId": first_file['file_id'],
        "ClientID": first_file['client_id'],
        "LayoutID": first_file['layout_id'],
        "ProcessType": "PROVIDER"
    }
    
    return json.dumps({"FileIds": [gold_entry]})

In [0]:
def trigger_gold_notebooks(gold_payload: str):
    """Triggers Gold layer notebook for dimension processing."""
    gold_notebooks_base = f"{ROOT_DIR}/dimProvider/Gold/Notebooks"
    
    try:
        print("\n=== Triggering GenericSubGroupProcessing (Gold Layer) ===")
        dbutils.notebook.run(
            f"{gold_notebooks_base}/GenericSubGroupProcessing", 
            600, 
            {"GoldPayload": gold_payload}
        )
        print("GenericSubGroupProcessing completed successfully\n")
    except Exception as e:
        print(f"Gold layer processing failed: {e}")
        raise

In [0]:
os.sync()

In [0]:
def main():
    """Main pipeline orchestration: Bronze → Silver → Gold"""
    # Check multiple source directories for provider-related files
    source_directories = [
        ROOT_DIR / "source/837/pending",  # Claims with provider info
        ROOT_DIR / "source/274/pending",  # Prior authorization/hierarchy
        ROOT_DIR / "source/provider/pending"  # Direct provider files
    ]
    
    # Collect all files from all directories
    incoming_files = []
    for pending_dir in source_directories:
        if pending_dir.exists():
            files = [f for f in pending_dir.iterdir() if f.is_file() and not f.name.startswith('.')]
            if files:
                print(f"Found {len(files)} file(s) in {pending_dir}")
                incoming_files.extend(files)
    
    if not incoming_files:
        print("No files found to process in any source directory.")
        return

    print(f"\nTotal files to process: {len(incoming_files)}")
    
    processed_files = []
    for file_path in incoming_files:
        try:
            # Determine the base source directory from the file path
            base_source_dir = file_path.parent.parent
            c_id, f_id, l_id, csv_filename = process_single_file(file_path, base_source_dir)
            processed_files.append({
                'client_id': c_id, 
                'file_id': f_id, 
                'layout_id': l_id,
                'csv_filename': csv_filename
            })
        except Exception as e:
            print(f"Skipping {file_path.name}: {e}")

    if not processed_files:
        print("No files were successfully processed.")
        return

    print(f"\nSuccessfully processed {len(processed_files)} file(s)")
    
    # Payload Generation & Orchestration Dispatches
    process_payload, consolidation_payload = build_payloads(processed_files)
    notebook_base = f"{ROOT_DIR}/Shared/Notebooks"
    
    try:
        print("\n" + "="*60)
        print("BRONZE LAYER PROCESSING")
        print("="*60)
        
        print("\n=== Triggering FilesToProcess ===")
        dbutils.notebook.run(f"{notebook_base}/FilesToProcess", 600, {"ProcessedJSON": process_payload})
        print("FilesToProcess completed successfully")
        
        print("\n=== Triggering LoopConsolidationFiles ===")
        dbutils.notebook.run(f"{notebook_base}/LoopConsolidationFiles", 600, {"ConsolidationJSON": consolidation_payload})
        print("LoopConsolidationFiles completed successfully")
        
        print("\n" + "="*60)
        print("SILVER LAYER PROCESSING")
        print("="*60)
        # Trigger Silver layer notebooks after consolidation
        trigger_silver_notebooks()
        
        print("\n" + "="*60)
        print("GOLD LAYER PROCESSING")
        print("="*60)
        # Build Gold payload and trigger Gold layer notebooks
        gold_payload = build_gold_payload(processed_files)
        trigger_gold_notebooks(gold_payload)
        
        print("\n" + "="*60)
        print("PIPELINE EXECUTION COMPLETED SUCCESSFULLY")
        print("Bronze -> Silver -> Gold")
        print("="*60)
    except Exception as e:
        print(f"\nDownstream orchestration failed: {e}")
        raise

if __name__ == "__main__":
    main()

In [0]:
# Direct execution - bypassing Bronze layer (files already processed)
print("=" * 60)
print("SILVER LAYER PROCESSING")
print("=" * 60)
trigger_silver_notebooks()

print("\n" + "=" * 60)
print("GOLD LAYER PROCESSING")
print("=" * 60)
# Use dummy payload since we're running directly
gold_payload = json.dumps({"FileIds": [{"FileId": "DIRECT_RUN", "ClientID": "SYSTEM", "LayoutID": "PROVIDER", "ProcessType": "PROVIDER"}]})
trigger_gold_notebooks(gold_payload)

print("\n" + "=" * 60)
print("SILVER AND GOLD LAYERS COMPLETED")
print("=" * 60)